In [25]:
import pandas as pd

# 1. Load datasets
patients = pd.read_csv("Patient_Data.csv")
billing = pd.read_csv("Billing_Data.csv")

In [26]:
# Show summary
print("Patient Dataset Info:")
print(patients.info())

Patient Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   PatientID       6 non-null      int64  
 1   Name            6 non-null      object 
 2   Department      6 non-null      object 
 3   Doctor          6 non-null      object 
 4   BillAmount      4 non-null      float64
 5   ReceptionistID  6 non-null      int64  
 6   CheckInTime     6 non-null      object 
dtypes: float64(1), int64(2), object(4)
memory usage: 468.0+ bytes
None


In [27]:
# 2. Select billing relevant columns
billing_columns = ['PatientID', 'Department', 'Doctor', 'BillAmount']
patients_billing = patients[billing_columns]

print(patients_billing)

   PatientID   Department     Doctor  BillAmount
0        101   Cardiology  Dr. Smith      5000.0
1        102    Neurology   Dr. John         NaN
2        103  Orthopedics    Dr. Lee      7500.0
3        104   Cardiology  Dr. Smith      6200.0
4        105  Dermatology   Dr. Rose         NaN
5        101   Cardiology  Dr. Smith      5000.0


In [28]:
# 3. Drop administrative columns
patients = patients.drop(columns=['ReceptionistID', 'CheckInTime'], errors='ignore')

print(patients)

   PatientID     Name   Department     Doctor  BillAmount
0        101    Alice   Cardiology  Dr. Smith      5000.0
1        102      Bob    Neurology   Dr. John         NaN
2        103  Charlie  Orthopedics    Dr. Lee      7500.0
3        104    David   Cardiology  Dr. Smith      6200.0
4        105      Eva  Dermatology   Dr. Rose         NaN
5        101    Alice   Cardiology  Dr. Smith      5000.0


In [29]:
# 4. Total bill amount per department
dept_bill = patients.groupby('Department')['BillAmount'].sum()
print("\nTotal Bill Amount per Department:")
print(dept_bill)


Total Bill Amount per Department:
Department
Cardiology     16200.0
Dermatology        0.0
Neurology          0.0
Orthopedics     7500.0
Name: BillAmount, dtype: float64


In [30]:
# 5. Remove duplicate patient records
patients = patients.drop_duplicates(subset='PatientID')

print(patients)

   PatientID     Name   Department     Doctor  BillAmount
0        101    Alice   Cardiology  Dr. Smith      5000.0
1        102      Bob    Neurology   Dr. John         NaN
2        103  Charlie  Orthopedics    Dr. Lee      7500.0
3        104    David   Cardiology  Dr. Smith      6200.0
4        105      Eva  Dermatology   Dr. Rose         NaN


In [31]:
# 6. Fill missing BillAmount with mean
mean_bill = patients['BillAmount'].mean()
patients['BillAmount'] = patients['BillAmount'].fillna(mean_bill)

print(patients['BillAmount'] )

0    5000.000000
1    6233.333333
2    7500.000000
3    6200.000000
4    6233.333333
Name: BillAmount, dtype: float64


In [32]:
# 7. Merge billing dataset with patient dataset
merged_df = pd.merge(patients, billing, on='PatientID', how='left')

print(merged_df)

   PatientID     Name   Department     Doctor   BillAmount  InsuranceCovered  \
0        101    Alice   Cardiology  Dr. Smith  5000.000000              2000   
1        102      Bob    Neurology   Dr. John  6233.333333              1500   
2        103  Charlie  Orthopedics    Dr. Lee  7500.000000              2500   
3        104    David   Cardiology  Dr. Smith  6200.000000              3000   
4        105      Eva  Dermatology   Dr. Rose  6233.333333              1000   

   FinalAmount  
0         3000  
1         3500  
2         5000  
3         3200  
4         4000  


In [33]:
# 8. Concatenate new patients (row-wise)
new_patients = pd.DataFrame({
    'PatientID': [9001, 9002],
    'Name' : ['Ava','jessy'],
    'Department': ['Cardiology', 'Neurology'],
    'Doctor': ['Dr. Kumar', 'Dr. Rao'],
    'BillAmount': [15000, 18000],
    
})

merged_df = pd.concat([merged_df, new_patients], axis=0, ignore_index=True)

print(merged_df)

   PatientID     Name   Department     Doctor    BillAmount  InsuranceCovered  \
0        101    Alice   Cardiology  Dr. Smith   5000.000000            2000.0   
1        102      Bob    Neurology   Dr. John   6233.333333            1500.0   
2        103  Charlie  Orthopedics    Dr. Lee   7500.000000            2500.0   
3        104    David   Cardiology  Dr. Smith   6200.000000            3000.0   
4        105      Eva  Dermatology   Dr. Rose   6233.333333            1000.0   
5       9001      Ava   Cardiology  Dr. Kumar  15000.000000               NaN   
6       9002    jessy    Neurology    Dr. Rao  18000.000000               NaN   

   FinalAmount  
0       3000.0  
1       3500.0  
2       5000.0  
3       3200.0  
4       4000.0  
5          NaN  
6          NaN  


In [37]:
# 9. Concatenate new billing category columns (column-wise)
extra_columns = pd.DataFrame({
    'InsuranceCovered': [True] * len(merged_df),
    'FinalAmount': merged_df['BillAmount'] * 0.9
})

final_df = pd.concat([merged_df, extra_columns], axis=1)

print(final_df)

   PatientID     Name   Department     Doctor    BillAmount  InsuranceCovered  \
0        101    Alice   Cardiology  Dr. Smith   5000.000000            2000.0   
1        102      Bob    Neurology   Dr. John   6233.333333            1500.0   
2        103  Charlie  Orthopedics    Dr. Lee   7500.000000            2500.0   
3        104    David   Cardiology  Dr. Smith   6200.000000            3000.0   
4        105      Eva  Dermatology   Dr. Rose   6233.333333            1000.0   
5       9001      Ava   Cardiology  Dr. Kumar  15000.000000               NaN   
6       9002    jessy    Neurology    Dr. Rao  18000.000000               NaN   

   FinalAmount  InsuranceCovered  FinalAmount  
0       3000.0              True       4500.0  
1       3500.0              True       5610.0  
2       5000.0              True       6750.0  
3       3200.0              True       5580.0  
4       4000.0              True       5610.0  
5          NaN              True      13500.0  
6          NaN 

In [36]:
print("\nFinal Cleaned Dataset:")
print(final_df.head())


Final Cleaned Dataset:
   PatientID     Name   Department     Doctor   BillAmount  InsuranceCovered  \
0        101    Alice   Cardiology  Dr. Smith  5000.000000            2000.0   
1        102      Bob    Neurology   Dr. John  6233.333333            1500.0   
2        103  Charlie  Orthopedics    Dr. Lee  7500.000000            2500.0   
3        104    David   Cardiology  Dr. Smith  6200.000000            3000.0   
4        105      Eva  Dermatology   Dr. Rose  6233.333333            1000.0   

   FinalAmount  InsuranceCovered  FinalAmount  
0       3000.0              True       4500.0  
1       3500.0              True       5610.0  
2       5000.0              True       6750.0  
3       3200.0              True       5580.0  
4       4000.0              True       5610.0  
